# Создание правильного набора данных.
Вообще по идее нашей задачи, нужно чтобы на вход подавался pdf (я еще проверю можно ли к ним закинуть чистый pdf, но я сомневаюсь довольно сильно. Щас заодно и проверим

In [26]:
from os import listdir

dataset_path = "/Users/mike/Lab/Dataset/theoremkb_arxiv/"

## Скачивание их датасета и предобработка
У меня скачивание полного набора занимает около 5 часов. Это довольно долго)

In [13]:
import os
import re
import time
import tarfile
import gzip
import shutil
import pandas as pd
import requests
from tqdm import tqdm


def extract_arxiv_id(link):
    """
    Извлекает arXiv ID из ссылок вида:
      - https://arxiv.org/abs/1309.1274
      - https://arxiv.org/abs/1309.1274v2
      - https://arxiv.org/pdf/1309.1274.pdf
      - https://arxiv.org/abs/math/0601001
      - https://arxiv.org/abs/hep-th/9901001v3
    Возвращает ID (без версии) или None, если не удалось.
    """
    if not isinstance(link, str):
        return None
    match = re.search(r"arxiv\.org/(?:abs|pdf)/([^\s?#]+)", link)
    if not match:
        return None
    raw = re.sub(r"\.pdf$", "", match.group(1))  # убираем .pdf
    raw = re.sub(r"v\d+$", "", raw)              # убираем версию vN
    raw = raw.rstrip("/")                        # убираем хвостовой слэш
    return raw if raw else None


def download_arxiv_sources_from_csv(
    csv_path: str,
    output_dir: str,
    delay_between_downloads: float = 3.0,
    overwrite: bool = False,
):
    # --- 1. Читаем CSV ---
    df = pd.read_csv(csv_path, header=None)
    link_col = None
    for col in df.columns:
        if df[col].astype(str).str.contains("arxiv.org", na=False).any():
            link_col = col
            break
    if link_col is None:
        raise ValueError("В CSV не найдена колонка со ссылками на arxiv.org")

    links = df[link_col].dropna().astype(str).tolist()
    print(f"Найдено {len(links)} ссылок в CSV.")

    # --- 2. Извлекаем arXiv ID ---
    arxiv_ids = []
    skipped_links = []
    for link in links:
        aid = extract_arxiv_id(link)
        if aid:                       # отсеиваем None и пустые строки
            arxiv_ids.append(aid)
        else:
            skipped_links.append(link)

    if skipped_links:
        print(f"⚠️ Не удалось распарсить ссылок: {len(skipped_links)}")
        for s in skipped_links[:10]:
            print(f"   {s}")

    # Дополнительная защита от None
    arxiv_ids = [x for x in arxiv_ids if isinstance(x, str) and x.strip()]

    if not arxiv_ids:
        print("❌ Не найдено ни одного валидного arXiv ID.")
        return {}

    # Дедупликация с сохранением порядка
    seen = set()
    arxiv_ids = [x for x in arxiv_ids if not (x in seen or seen.add(x))]
    print(f"Уникальных arXiv ID: {len(arxiv_ids)}")

    # --- 3. Создаём выходную папку ---
    os.makedirs(output_dir, exist_ok=True)

    results = {}

    # --- 4. Скачиваем ---
    for arxiv_id in tqdm(arxiv_ids, desc="Скачивание статей"):
        # Безопасное имя папки (заменяем "/" на "_" для старых ID)
        safe_id = arxiv_id.replace("/", "_")
        target_dir = os.path.join(output_dir, safe_id)

        if os.path.exists(target_dir) and not overwrite:
            results[arxiv_id] = "skipped (already exists)"
            continue

        if os.path.exists(target_dir) and overwrite:
            shutil.rmtree(target_dir)

        os.makedirs(target_dir, exist_ok=True)

        url = f"https://arxiv.org/e-print/{arxiv_id}"
        headers = {
            "User-Agent": "TheoremKB-Downloader/1.0 (contact: your-email@example.com)"
        }

        try:
            response = requests.get(url, headers=headers, timeout=60, stream=True)
            response.raise_for_status()

            tmp_archive = os.path.join(target_dir, f"{safe_id}.tar.gz")
            with open(tmp_archive, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)

            try:
                with tarfile.open(tmp_archive, "r:gz") as tar:
                    for member in tar.getmembers():
                        member_path = os.path.join(target_dir, member.name)
                        if not os.path.abspath(member_path).startswith(os.path.abspath(target_dir)):
                            raise Exception(f"Небезопасный путь: {member.name}")
                    tar.extractall(path=target_dir)
                results[arxiv_id] = "success (tar.gz)"
            except tarfile.ReadError:
                try:
                    with gzip.open(tmp_archive, "rb") as gz:
                        content = gz.read()
                    with open(os.path.join(target_dir, f"{safe_id}.tex"), "wb") as f:
                        f.write(content)
                    results[arxiv_id] = "success (single .tex.gz)"
                except Exception as e:
                    results[arxiv_id] = f"error: распаковка: {e}"

            if os.path.exists(tmp_archive):
                os.remove(tmp_archive)

        except requests.HTTPError as e:
            results[arxiv_id] = f"error: HTTP {e.response.status_code}"
        except Exception as e:
            results[arxiv_id] = f"error: {e}"

        time.sleep(delay_between_downloads)

    # --- 5. Отчёт ---
    print("\n" + "=" * 60)
    print("ИТОГИ СКАЧИВАНИЯ")
    print("=" * 60)
    ok = sum(1 for v in results.values() if v.startswith("success"))
    skipped = sum(1 for v in results.values() if v.startswith("skipped"))
    errors = sum(1 for v in results.values() if v.startswith("error"))
    print(f"✅ Успешно: {ok}")
    print(f"⏭️  Пропущено: {skipped}")
    print(f"❌ Ошибок: {errors}")

    if errors:
        print("\nОшибки по ID:")
        for k, v in results.items():
            if v.startswith("error"):
                print(f"  {k}: {v}")

    return results


# --- Вызов ---
download_arxiv_sources_from_csv(
    csv_path="links.csv",
    output_dir="/Users/mike/Lab/Dataset/theoremkb_arxiv/",
    delay_between_downloads=3.0,
    overwrite=False,
)

Найдено 4401 ссылок в CSV.
⚠️ Не удалось распарсить ссылок: 1
   urls
Уникальных arXiv ID: 4400


Скачивание статей:   5%|▍         | 205/4400 [15:19<5:13:29,  4.48s/it] 


KeyboardInterrupt: 

In [22]:
import shutil
for folder in os.listdir(dataset_path):
    path = os.path.join(dataset_path, folder)
    shutil.copy("util_files/extthm.sty", path)


### Преобразование latex в pdf
Возможно на этом этапе получится вставить другие pdf для обработки, но пока что не факт
Также нужно специальный файл стилей *.sty
С помощью него как-то идет предварительный парсинг статей, чтобы получить правильные ответы.

In [27]:
from util_scripts.tex2pdf import tex2pdf
res = tex2pdf(
    source_directory=dataset_path,
    new_extthm_path="util_files/extthm.sty",
    n_jobs=4,
).fit()

error-- no style file found







100%|██████████| 204/204 [00:00<00:00, 8401.95it/s]





injecting extthm: 100%|██████████| 204/204 [00:00<00:00, 30842.69it/s][A


no \documentclass found in /Users/mike/Lab/Dataset/theoremkb_arxiv/1210.5677/1210.5677.tex







  0%|          | 0/204 [00:00<?, ?it/s][Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.





  4%|▍         | 8/204 [00:00<00:09, 20.68it/s][Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.5s





  8%|▊         | 16/204 [00:00<00:06, 30.74it/s][Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Batch computation too fast (0.19128983603496272s.) Setting batch_size=2.





 12%|█▏        | 24/204 [00:00<00:04, 38.36it/s][Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    0.8s





 20%|█▉        | 40/204 [00:00<00:03, 53.55it/s]




 24%|██▎       | 48/204 [00:01<00:02, 54.77it/s][Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.2s





 27%|██▋       | 56/204 [00:01<00:02, 52.82it/s]




 31%|███▏      | 64/204 [00:01<00:02, 54.31it/s]




 35%|███▌      | 72/204 [00:01<00:02, 56.30it/s][Parallel(n_jobs=4)]: Done  60 tasks      | elapsed:    1.

### Перенос готовых pdf в новую папку
По идее дальше все что будет использоваться это pdflato+grobid. Можно сюда попробовать добавить наши pdf но насколько я знаю там какой-то прикол с файлом стилей, который мы импортируем, что типо он создает разметку. Но если нам разметка не сильно нужна то пох наверное

In [28]:
clear_dataset_theoremkbpath = "/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/"

In [29]:

from util_scripts.copy_dest import copy_dest
copy_dest(source=dataset_path, destination=clear_dataset_theoremkbpath).copy()






  0%|          | 0/204 [00:00<?, ?it/s]




 41%|████      | 83/204 [00:00<00:00, 825.69it/s]




100%|██████████| 204/204 [00:00<00:00, 939.54it/s]


Здесь нужен будет Grobid. По скрипт настроен на то, чтобы использовать его из докера, а не запускать автоматически.
Ячейчка в каждом каталоге создат файл <<Название статьи>>.tei.xml. Это базовая разметка файла вообще конечно в идеале проверить, как работает в связке с MinerU и Pager например, но при создании датасета получается так что гробит очень нужен, поскольку формируются правильные аннотации для блоков (именно для обучения очень надо). Дообучать языковые модели мы наверное не будем, но применить такой подход можно.



Некоторые документы прямо долго парсятся, возможна такая ситуация что очень долго обрабатывается. В таком случае надо поставить в файле util_scripts/grobid_client_python/config.json поставить подольше чтобы точно все обработались.

In [ ]:
%%sql


In [5]:
import os
grobid_client = "/Users/mike/grobid-client-python-0.2.0" #https://github.com/grobidOrg/grobid-client-python/releases/tag/v0.2.0 Просто скачать и положить в папку, ниче больше делать не надо


sub_folders = [
    os.path.join(clear_dataset_theoremkbpath, element) for element in os.listdir(clear_dataset_theoremkbpath)
]

from util_scripts.grobid_on_pdfs import grobid_on_pdfs

grobid_on_pdfs(
    grobid_client=grobid_client,
    sub_folders=sub_folders,
    n_jobs=4,
).fit()




Проверка папок: 100%|██████████| 205/205 [00:00<00:00, 19084.06it/s]


ℹ️  Контейнер 'grobid' уже запущен.
✅ Grobid готов.
GROBID server is up and running


Grobid processing:   8%|▊         | 16/205 [04:31<53:44, 17.06s/it]

Processing of /Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1104.0185/BG05_reform.pdf failed with error 408


Grobid processing:  18%|█▊        | 36/205 [09:58<43:56, 15.60s/it]  

...

Processing of /Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1002.3864/dimacs_limits.pdf failed with error 408


Grobid processing:  90%|████████▉ | 184/205 [1:05:05<06:36, 18.90s/it]

Processing of /Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1708.08179/hard_presburger3.pdf failed with error 408


Grobid processing: 100%|██████████| 205/205 [1:11:29<00:00, 20.92s/it]


Processing of /Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1407.7763/icm.pdf failed with error 408
Processing of /Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1404.1689/Potential_of_quantum_finite_automata_with_exact_acceptance_arxiv_20141125.pdf failed with error 408
🛑 Останавливаю контейнер Grobid...


Проверка, сколько файлов обработалось

In [11]:
import glob
from shutil import rmtree
from tqdm import tqdm

grobid_files = glob.glob(f"{clear_dataset_theoremkbpath}/**/**.tei.xml")
len(grobid_files)


156

Видимо проверка на то, как обработались pdf-ки. Однако тут используется довольно серьезный скрипт, который анализирует гробид файлы.
Также если просмотреть какой-либо файл с помощью этого скрипта с show_results=True, создастся папка с изображениями pdf, которые хз зачем нужны. Видимо чтобы показывать как раз bbox, которые выделил гробид.

In [18]:
from util_scripts.grobid_clean import Preprocess_using_grobid
import time
article_name_example = "0905.4100"
path = os.path.join(clear_dataset_theoremkbpath, article_name_example)
for file in os.listdir(path):
    if file.endswith(".tei.xml"):
       tei_path = os.path.join(path, file)
print(tei_path)
tick = time.time()
prep = Preprocess_using_grobid()
final = prep.fit(
    # grobid_xml="/Users/mike/articles/stat/A1-Jacobi-Dif-corrected.tei.xml",
    grobid_xml=tei_path,
    show_results=True,
)
tock = time.time()
print(tock - tick)


/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/0905.4100/0905.4100.tei.xml
generating visualizations
inside_visualizations


100%|██████████| 15/15 [00:05<00:00,  2.88it/s]


Introduction
[2, (200.0, 290.8333333333333), (1479.9999999999998, 392.3888888888889), 'Bipartite matching problems are central in combinatorial optimization with many applications.Our motivating application is the allocation of display advertisements on the Internet, 1 and so we will use the language of this application to define and discuss the problem:']
[2, (200.0, 422.27777777777777), (1480.0000000000002, 600.0555555555555), '(Online Bipartite Matching) There is a bipartite graph G(A, I, E) with advertisers A and impressions I, and a set E of edges between them.Advertisers in A are fixed and known.Impressions (or requests) in I (along with their incident edges) arrive online.Upon the arrival of an impression i ∈ I, we must assign i to any advertiser a ∈ A where (i, a) ∈ E(G).At all times, the set of assigned edges must form a matching (that is, no end points coincide).']
[2, (200.0, 629.9444444444445), (1480.0277777777776, 770.111111111111), 'If the online algorithm knows nothing a

 ### PDFALTO

Тут нужно скачать pdfalto. Вообще это еще один обработчик pdf, который они используют. Непонятно если честно зачем они второй раз прогоняют через парсер, ну посмотрим что выйдет.
Скачать нужно бинарник исполняемый, а потом просто передать ссылку на файл где он находится.

In [11]:
from util_scripts.pdfalto_on_pdfs import pdfalto_on_pdfs
import os

pdfalto = "/Users/mike/Downloads/pdfalto"

pdfs = pdfalto_on_pdfs(pdfalto=pdfalto, n_jobs=4, main_folder=clear_dataset_theoremkbpath).fit()

204


 63%|██████▎   | 128/204 [00:07<00:04, 16.62it/s]Syntax Error (396863): Missing 'endstream'
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in CCITTFax stream
Syntax Error (393044): Bad two dim code (0001) in 

Вот тут можно посмотреть, что сделал pdfalto с аннотациями. ВОобще непонятно тогда зачем нужен гробид, если мы обрабатываем все равно pdfalto.
upd: стало понятно) они потом как то объеденяют выводы, чтобы получить типо единые данные. pdfalto дает аннотации, grobid структуру, они потом совмещают все это дело и получаются данные.

In [12]:
from util_scripts.visualize_annot import annotations_page
# try this - /Users/mv96/Desktop/temp/1902.11202/1902.11202_annot.xml
# /Users/mv96/Desktop/temp/1910.12458/1910.12458_annot.xml
annot = annotations_page(
    labels_xml="/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/0908.4016/0908.4016_annot.xml",
    show_images=True,
)

annotations = annot.fit()


100%|██████████| 5/5 [00:00<00:00, 25.10it/s]


uri:extthm.theorem.1
[2, (300.0, 448.37499999999994), (1420.0, 515.4833333333333), 'uri:extthm.theorem.2']
[1, (300.0, 1795.675), (1420.0, 1888.2305555555556), 'uri:extthm.theorem.1']
[2, (300.0, 448.37499999999994), (1420.0, 515.4833333333333), 'uri:extthm.theorem.2']
[1, (300.0, 1795.675), (1420.0, 1888.2305555555556), 'uri:extthm.theorem.1']
[2, (300.0, 448.37499999999994), (1420.0, 515.4833333333333), 'uri:extthm.theorem.2']
[2, (300.0, 878.7055555555555), (1420.0, 974.7194444444444), 'uri:extthm.theorem.3']
[2, (300.0, 1462.011111111111), (1420.0, 1553.0277777777778), 'uri:extthm.theorem.4']
[2, (300.0, 1589.3111111111111), (1420.0, 1887.6527777777776), 'uri:extthm.proof.1']
[3, (300.0, 353.5916666666667), (1420.0, 928.7694444444445), 'uri:extthm.proof.1']
[2, (300.0, 1589.3111111111111), (1420.0, 1887.6527777777776), 'uri:extthm.proof.1']
[3, (300.0, 353.5916666666667), (1420.0, 928.7694444444445), 'uri:extthm.proof.1']
[4, (300.0, 614.4694444444444), (1420.0, 705.4861111111111),

 Тут каждому блоку, который нашел гробид, присваивается класс. Конкретно как происходит присвоение класса описано в скрипте (там на самом деле не сильно сложно)

In [6]:
# try=/Users/mv96/new/dest-pdfs/1509.06361/file.tei.xml
from util_scripts.labelling import assigning_labels

# name 1712.06239/main.pdf
grobid_xml = "/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/0908.4016/0908.4016.tei.xml"  # grobid xml
labels_xml = "/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/0908.4016/0908.4016_annot.xml"  # annotations xml

table, scales = assigning_labels(
    show_images=False, grobid_xml=grobid_xml, labels_xml=labels_xml
).fit()

table


100%|██████████| 5/5 [00:00<00:00, 37.42it/s]


,page_no,top_left,bot_right,text,label
0,1,"(300.0, 1047.6944444444443)","(613.9722222222222, 1082.75)",Introduction,basic
1,1,"(300.0, 1116.7499999999998)","(1420.0277777777776, 1174.2222222222222)","Throughout this paper G = (V, E) is a simple (...",basic
2,1,"(341.5, 1183.1666666666665)","(961.5277777777777, 1209.0)","Let S ⊆ V be a set of vertices, and let i ∈ N....",basic
3,1,"(626.6944444444445, 1244.0555555555557)","(1093.3333333333333, 1270.861111111111)","N i (S) = {w ∈ V | min s∈S d(w, s) = i},",basic
4,1,"(300.0, 1304.9444444444443)","(1420.0, 1329.2222222222222)","where d(x, y) is the minimal number of edges r...",basic
...,...,...,...,...,...
62,4,"(341.5, 1469.6666666666667)","(671.2222222222222, 1496.4722222222222)",The directed edges E F are:,uri:extthm.proof.2
63,4,"(341.52777777777777, 1527.8888888888887)","(978.9444444444445, 1556.5555555555554)",• the directed edges from s to each vertex of ...,uri:extthm.proof.2
64,4,"(341.52777777777777, 1637.972222222222)","(1122.75, 1666.638888888889)","• the directed edges va i , for each 1 ≤ i ≤ k...",uri:extthm.proof.2
65,4,"(341.52777777777777, 1693.0)","(872.4722222222222, 1721.6666666666667)","• the directed edges a i t, for each 1 ≤ i ≤ k.",uri:extthm.proof.2


Проверка на валидные документы. Скрипт просто смотрит, есть ли файлы нужные на месте, а потом проверяет есть ли хотя бы одна аннотация. Если есть значит все сработало и pdf считается валидным.

In [30]:
# 5 filtering the outputs generated using the annot_file
from util_scripts.pdf_is_valid import pdf_is_valid

pos, neg = pdf_is_valid(clear_dataset_theoremkbpath, n_jobs=4).fit()

print(len(pos), len(neg))
# 2665 858

with open("negative_pdfs.txt", "w") as f:
    for sample in neg:
        f.write(sample + "\n")

f.close()






  0%|          | 0/109 [00:00<?, ?it/s]




 37%|███▋      | 40/109 [00:00<00:00, 277.32it/s]




100%|██████████| 109/109 [00:00<00:00, 371.94it/s][A


109 0


Удаление тех которые невалидные

In [32]:
# we can remove the remaining ones
from shutil import rmtree

for files in neg:
    rmtree(files.rsplit("/", 1)[0])

### Создает массив файлов, которые идут в кучке: разметка grobid, разметка c аннотациями, разметка xml, сам пдф

In [35]:
sets = []
for pdf_file in pos:
    grobid_xml = pdf_file.replace(".pdf", ".tei.xml")
    labels_xml = pdf_file.replace(".pdf", "_annot.xml")
    pdf_alto_xml_main = pdf_file.replace(".pdf", ".xml")
    sample_pdf = pdf_file
    sets.append((grobid_xml, labels_xml, pdf_alto_xml_main, sample_pdf))

print(len(sets))
print(sets[0])

109
('/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1806.06973/main.tei.xml', '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1806.06973/main_annot.xml', '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1806.06973/main.xml', '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1806.06973/main.pdf')


Это вообще циганский ход конем. Здесь в эксель файле содержится ручная разметка шрифтов. Ну то есть прямо перечислены шрифты и написан номер класса к которому его надо отнести.

На этом этапе в каждой папке появился файлик data.csv, который составлен из каждой работы. В нем уже типо полные данные со всеми признаками, которые удалось выделить из статьи.

In [36]:
from util_scripts.dataframe_for_eval import preprocessed_dataframe

excel_file = "util_files/output.xlsx"  # the path xlsx of the manually labelled fonts


prep = preprocessed_dataframe(sets=sets, excel_file=excel_file, n_jobs=-2)

prep.fit()







  0%|          | 0/109 [00:00<?, ?it/s][Parallel(n_jobs=-2)]: Using backend LokyBackend with 9 concurrent workers.





  1%|          | 1/109 [00:00<00:14,  7.23it/s]/Users/mike/PycharmProjects/theoremkb/Dataset/util_scripts/grobid_clean.py:849: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
  second_cmd = "LC_ALL=C sed -n '/^\/Annots/!p' {} > {}".format(
/Users/mike/PycharmProjects/theoremkb/Dataset/util_scripts/grobid_clean.py:849: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
  second_cmd = "LC_ALL=C sed -n '/^\/Annots/!p' {} > {}".format(
/Users/mike/PycharmProjects/theoremkb/Dataset/util_scripts/grobid_clean.py:849: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
  second_cmd = "L

[SKIP] /Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1409.3836/HardnessLearningParams.tei.xml: TypeError: can only concatenate list (not "int") to list


 33%|███▎      | 4/12 [00:00<00:00, 35.16it/s]

[SKIP] /Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1509.05572/1509.05572.tei.xml: AssertionError: 


100%|██████████| 12/12 [00:00<00:00, 38.06it/s]
[Parallel(n_jobs=-2)]: Done 109 out of 109 | elapsed:   22.4s finished


[('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1806.06973/main.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1911.04112/main.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1811.05115/LatestVersion.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1007.1073/1007.1073.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1809.02835/multitaskingitcs.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1002.4482/ParGraphAlgOnGPU-Paper-Report.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1801.04497/sim-maxcut.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1811.08097/ms.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clear/1312.1718/uppersemicomputable.tei.xml',
  None),
 ('skipped',
  '/Users/mike/Lab/Dataset/theoremkb_arxiv_clea